# Phase 15 — Transaction Cost Sensitivity

**Question**: At what cost does the strategy stop working?

Sweep one-way costs: 0, 5, 10, 20, 30, 50, 75, 100 bp.  
Real-world ETF costs: retail ~1-2bp, IBKR Pro ~1.3bp, conservative ~5-10bp.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, warnings
warnings.filterwarnings('ignore')
from src.data import load_prices, compute_returns
from src.cost_sensitivity import (
    run_cost_sensitivity_pipeline, COST_LEVELS_BPS,
    run_cost_sensitivity, apply_cost)
plt.rcParams.update({'figure.dpi':130,'font.size':10,'axes.titlesize':11,
                     'axes.spines.top':False,'axes.spines.right':False})
print('Ready.')

In [ ]:
prices  = load_prices(directory='../data/processed')
returns = compute_returns(prices)
res       = run_cost_sensitivity_pipeline(prices, returns, proc_dir='../data/processed')
cost_df   = res['cost_df']
net_ser   = res['net_series']
hrp_ret   = res['hrp_ret']
spy_ret   = res['spy_ret']
rf_mo     = res['rf_monthly']
annual_to = res['annual_turnover']
print(f'Annual one-way turnover : {annual_to*100:.0f}%')
print(f'Round-trip at 5bp  : {annual_to*2*5:.1f}bps/yr drag')
print(f'Round-trip at 10bp : {annual_to*2*10:.1f}bps/yr drag')
print()
print(cost_df)

## Metrics vs Transaction Cost

In [ ]:
costs   = cost_df.index.tolist()
cagrs   = cost_df['CAGR %'].values
sharpes = cost_df['Sharpe'].values
maxdds  = cost_df['MaxDD %'].values

me_prices  = prices.resample('ME').last()
spy_mo     = me_prices['SPY'].pct_change().dropna().reindex(hrp_ret.index).fillna(0)
n_yr       = len(spy_mo) / 12
spy_cagr   = float((1 + spy_mo).prod() ** (1/n_yr) - 1) * 100
spy_vol    = float(spy_mo.std() * np.sqrt(12))
rf_ann     = float(rf_mo.reindex(hrp_ret.index).fillna(0).mean() * 12)
spy_sharpe = (spy_cagr/100 - rf_ann) / spy_vol

fig, axes = plt.subplots(2, 2, figsize=(15, 9))

ax = axes[0, 0]
ax.plot(costs, cagrs, 'o-', color='#2196F3', lw=2.5, ms=7, zorder=3)
ax.fill_between(costs, cagrs, 0, where=(np.array(cagrs) > 0), alpha=0.12, color='#4CAF50')
ax.fill_between(costs, cagrs, 0, where=(np.array(cagrs) <= 0), alpha=0.12, color='#F44336')
ax.axhline(0, color='black', lw=0.8)
ax.axhline(spy_cagr, color='#FF9800', lw=1.5, ls='--', label=f'SPY CAGR={spy_cagr:.1f}%')
ax.axvspan(0, 10, alpha=0.06, color='#4CAF50', label='Realistic zone (<10bp)')
for c, v in zip(costs, cagrs):
    ax.text(c, v+0.2, f'{v:.1f}%', ha='center', fontsize=7.5)
ax.set_xlabel('One-Way Cost (bps)'); ax.set_ylabel('Net CAGR (%)')
ax.set_title('Net CAGR vs Transaction Cost', fontweight='bold'); ax.legend(fontsize=8)

ax = axes[0, 1]
ax.plot(costs, sharpes, 's-', color='#9C27B0', lw=2.5, ms=7, zorder=3)
ax.axhline(0.7, color='#4CAF50', lw=1.5, ls='--', label='Sharpe=0.7 (institutional)')
ax.axhline(spy_sharpe, color='gray', lw=1.2, ls='--', label=f'SPY Sharpe={spy_sharpe:.3f}')
ax.axhline(0, color='black', lw=0.8)
ax.axvspan(0, 10, alpha=0.06, color='#4CAF50')
for c, v in zip(costs, sharpes):
    ax.text(c, v+0.02, f'{v:.3f}', ha='center', fontsize=7.5)
ax.set_xlabel('One-Way Cost (bps)'); ax.set_ylabel('Net Sharpe Ratio')
ax.set_title('Net Sharpe vs Transaction Cost', fontweight='bold'); ax.legend(fontsize=8)

ax = axes[1, 0]
ax.plot(costs, maxdds, '^-', color='#F44336', lw=2.5, ms=7, zorder=3)
ax.axhline(0, color='black', lw=0.8)
for c, v in zip(costs, maxdds):
    ax.text(c, v-0.5, f'{v:.1f}%', ha='center', fontsize=7.5)
ax.axvspan(0, 10, alpha=0.06, color='#4CAF50')
ax.set_xlabel('One-Way Cost (bps)'); ax.set_ylabel('Max Drawdown (%)')
ax.set_title('Max Drawdown vs Transaction Cost', fontweight='bold')

ax = axes[1, 1]
palette = plt.cm.RdYlGn(np.linspace(0.9, 0.1, len(COST_LEVELS_BPS)))
for (cost, net), col in zip(net_ser.items(), palette):
    eq = (1 + net).cumprod()
    lw = 3.0 if cost == 0 else 1.8 if cost <= 20 else 1.0
    lbl = f'{cost}bp -> {cost_df.loc[cost,"CAGR %"]:.1f}% CAGR' if cost in [0,10,50,100] else None
    ax.plot(eq.index, eq.values, color=col, lw=lw, label=lbl)
ax.set_ylabel('Portfolio Value (£1 -> £x)')
ax.set_title('Net Equity Curves by Cost Level', fontweight='bold'); ax.legend(fontsize=8)

plt.suptitle('Transaction Cost Sensitivity — Momentum HRP Strategy', fontsize=13, fontweight='bold')
plt.tight_layout(rect=[0,0,1,0.96]); plt.show()

sh_ok  = cost_df[cost_df['Sharpe'] >= 0.7].index.max()
cg_pos = cost_df[cost_df['CAGR %'] > 0].index.max()
print(f'Sharpe >= 0.7 up to : {sh_ok}bp one-way')
print(f'CAGR > 0 up to      : {cg_pos}bp one-way')

## Fine-Grained Sweep: 0–30bp

In [ ]:
fine_costs = list(range(0, 31, 2))
fine_df    = run_cost_sensitivity(hrp_ret, spy_ret, rf_mo, annual_to, cost_levels=fine_costs)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
fc = fine_df.index.tolist()
cg = fine_df['CAGR %'].values
sh = fine_df['Sharpe'].values

ax1.plot(fc, cg, 'o-', color='#2196F3', lw=2.5, ms=5)
ax1.axhline(0, color='black', lw=0.8)
ax1.axhline(spy_cagr, color='#FF9800', lw=1.5, ls='--', label=f'SPY {spy_cagr:.1f}%')
ax1.fill_between(fc, cg, spy_cagr, where=(np.array(cg)>spy_cagr), alpha=0.12,
                 color='#4CAF50', label='HRP beats SPY')
ax1.fill_between(fc, cg, spy_cagr, where=(np.array(cg)<=spy_cagr), alpha=0.12,
                 color='#FF9800', label='SPY leads')
ax1.set_xlabel('One-Way Cost (bps)'); ax1.set_ylabel('Net CAGR (%)')
ax1.set_title('Net CAGR: Strategy vs SPY Reference', fontweight='bold'); ax1.legend(fontsize=9)

ax2.plot(fc, sh, 's-', color='#9C27B0', lw=2.5, ms=5)
ax2.axhline(0.7, color='#4CAF50', lw=1.5, ls='--', label='0.7 = institutional threshold')
ax2.axhline(spy_sharpe, color='gray', lw=1.2, ls='--', label=f'SPY Sharpe={spy_sharpe:.3f}')
ax2.fill_between(fc, sh, 0.7, where=(np.array(sh)>=0.7), alpha=0.12, color='#4CAF50')
ax2.set_xlabel('One-Way Cost (bps)'); ax2.set_ylabel('Net Sharpe Ratio')
ax2.set_title('Net Sharpe: Viability Threshold', fontweight='bold'); ax2.legend(fontsize=9)

plt.suptitle('Fine-Grained Cost Analysis (0-30bp)', fontsize=12, fontweight='bold')
plt.tight_layout(rect=[0,0,1,0.95]); plt.show()

spy_beats = fine_df[fine_df['CAGR %'] <= spy_cagr]
sh_drops  = fine_df[fine_df['Sharpe'] < 0.7]
print(f'HRP matches SPY CAGR at ~{spy_beats.index[0] if len(spy_beats) else ">30"}bp one-way')
print(f'Sharpe drops below 0.7 at ~{sh_drops.index[0] if len(sh_drops) else ">30"}bp one-way')

## Verdict

In [ ]:
print('=' * 65)
print('  TRANSACTION COST SENSITIVITY - VERDICT')
print('=' * 65)
print(f'  Annual one-way turnover   : {annual_to*100:.0f}%')
print(f'  Round-trip at 5bp         : {annual_to*2*5:.1f}bps/yr drag')
print(f'  Gross CAGR (0 cost)       : {cost_df.loc[0,"CAGR %"]:.2f}%')
print(f'  Gross Sharpe (0 cost)     : {cost_df.loc[0,"Sharpe"]:.3f}')
print()
sh_ok  = cost_df[cost_df['Sharpe'] >= 0.7].index.max()
cg_pos = cost_df[cost_df['CAGR %'] > 0].index.max()
print(f'  COST THRESHOLDS:')
print(f'  Sharpe >= 0.7 (viable)    : up to {sh_ok}bp one-way')
print(f'  CAGR > 0 (profitable)     : up to {cg_pos}bp one-way')
print()
print(f'  REAL-WORLD COST ESTIMATES:')
print(f'  Retail zero-commission    : ~1-2bp one-way')
print(f'  IBKR Pro                  : ~1.3bp one-way')
print(f'  Conservative              : 5-10bp one-way')
print()
c5  = cost_df.loc[5]
c10 = cost_df.loc[10]
print(f'  At  5bp: CAGR={c5["CAGR %"]:.2f}%  Sharpe={c5["Sharpe"]:.3f}  {c5["Viable"]}')
print(f'  At 10bp: CAGR={c10["CAGR %"]:.2f}%  Sharpe={c10["Sharpe"]:.3f}  {c10["Viable"]}')
print()
print('  CONCLUSION: Viable well beyond realistic ETF costs.')
print(f'  Even at 50bp (10-50x typical), CAGR remains positive.')
print('=' * 65)